In [ ]:
!pip -q install qiskit

In [ ]:
!pip -q install pennylane

In [ ]:
import numpy as np
import pandas as pd

First, let's prepare some sample data points to use for our kernel matrix calculation. We'll use 5 data points, each with 4 features, similar to your `phi` array.

Now, we'll define a function to compute a single kernel element (similarity) between two data points using your `circuit`'s output. A simple approach is the dot product of the expectation value vectors.

In [ ]:
num_data_points = 5
num_features = 4
# Generate random data points for demonstration
sample_data = np.random.rand(num_data_points, num_features)
print("Sample Data Points:")
print(type(sample_data))
print(sample_data)

Sample Data Points:
<class 'numpy.ndarray'>
[[0.79860762 0.65999114 0.47763421 0.10118533]
 [0.73948878 0.10231874 0.25885624 0.70714634]
 [0.85405058 0.08870851 0.33582856 0.56772695]
 [0.59690563 0.95162283 0.03785306 0.64735387]
 [0.68856563 0.53440765 0.5768747  0.25190744]]


In [ ]:
import pennylane as qml # Added pennylane import
import math # Added math import

dev = qml.device("default.mixed", wires=4) # Define the device

@qml.qnode(dev) #Define a QNode with measurements
def circuit(phi):
  for j in range(4):
    qml.RY(math.pi*phi[j], wires = j) #encoding data

    # Q Circuit - Quantum convolution
    qml.CRot(math.pi/4, 0, math.pi/4, wires=[0,1])
    qml.CRot(math.pi/4, 0, math.pi/4, wires=[1,2]) # control q and target qubit
    qml.CRot(math.pi/4, 0, math.pi/4, wires=[2,3])

    # Measurement = "updated features"
    return [qml.expval(qml.PauliZ(j)) for j in range(4)]

def quantum_kernel_element(x1, x2):
    # Run the circuit for each data point to get their feature vectors
    f_x1 = circuit(x1)
    f_x2 = circuit(x2)

    # Convert lists to numpy arrays for dot product
    f_x1_np = np.array(f_x1)
    f_x2_np = np.array(f_x2)

    # Calculate the dot product of the feature vectors
    return np.dot(f_x1_np, f_x2_np)

# Test the kernel element function with two sample data points
k_val = quantum_kernel_element(sample_data[0], sample_data[1])
print(f"Kernel value between data point 0 and 1: {k_val}")

Kernel value between data point 0 and 1: 3.551100035719894


In [ ]:
num_data_points = 5
num_features = 4
# Generate random data points for demonstration
sample_data = np.random.rand(num_data_points, num_features)
print("Sample Data Points:")
print(sample_data)

Sample Data Points:
[[0.97090322 0.84209478 0.18477034 0.79185618]
 [0.94545983 0.05203471 0.9586277  0.62329775]
 [0.42680593 0.64522703 0.76952626 0.71147503]
 [0.12772112 0.30040921 0.95046583 0.85378144]
 [0.43608921 0.48779709 0.84470455 0.04986386]]


Finally, let's construct the full kernel matrix by computing the `quantum_kernel_element` for all pairs of our `sample_data`.

In [ ]:
kernel_matrix = np.zeros((num_data_points, num_data_points))

for i in range(num_data_points):
    for j in range(num_data_points):
        kernel_matrix[i, j] = quantum_kernel_element(sample_data[i], sample_data[j])

print("Computed Kernel Matrix:")
print(kernel_matrix)

Computed Kernel Matrix:
[[3.65034245 3.55110004 3.72314225 3.24173553 3.45027528]
 [3.55110004 3.46700204 3.61279057 3.20484663 3.38156316]
 [3.72314225 3.61279057 3.80409131 3.26879558 3.50067942]
 [3.24173553 3.20484663 3.26879558 3.0898543  3.16736957]
 [3.45027528 3.38156316 3.50067942 3.16736957 3.31175549]]
